# 02 - Generate Data

## Team Challenge SQL - E-commerce tecnológico





1. Importamos las dependencias

In [4]:
import os
import random
import uuid
from pathlib import Path
from datetime import date, timedelta
from decimal import Decimal

import pandas as pd
from faker import Faker
from dotenv import load_dotenv

from google.cloud import bigquery
from google.oauth2 import service_account

print("Imports OK")

Imports OK


2. Comprobamos las conexiones

In [ ]:
PROJECT_ROOT = Path(
    r"C:\Users\casco\Desktop\bootcamp\Entregas\SQLMurderMystery"
)

ENV_PATH = PROJECT_ROOT / ".env"

CREDENTIALS_PATH = (
    PROJECT_ROOT
    / "credentials"
    / "service-account.json"
)

load_dotenv(ENV_PATH)

PROJECT_ID = os.getenv("GCP_PROJECT_ID")
DATASET_ID = os.getenv("BQ_DATASET_ID")

credentials = (
    service_account
    .Credentials
    .from_service_account_file(
        str(CREDENTIALS_PATH)
    )
)

client = bigquery.Client(
    project=PROJECT_ID,
    credentials=credentials
)

print("Conexión correcta")
print("Proyecto:", PROJECT_ID)
print("Dataset:", DATASET_ID)
print("Credenciales:", CREDENTIALS_PATH)


✅ Conexión correcta
Proyecto: 1057049326827
Dataset: amisbeauty
Credenciales: C:\Users\casco\Desktop\bootcamp\Entregas\SQLMurderMystery\credentials\service-account.json
Existe: True


3. Configuramos cómo se van a a generar los datos falsos

In [ ]:
# Semilla para que los datos sean reproducibles
SEED = 42

random.seed(SEED)

fake = Faker("es_ES")
fake.seed_instance(SEED)

# Número de registros a generar
N_CUSTOMERS = 500
N_CATEGORIES = 10
N_PRODUCTS = 70
N_ORDERS = 2000
N_ORDER_ITEMS = 4500

print(f"Clientes: {N_CUSTOMERS}")
print(f"Categorías: {N_CATEGORIES}")
print(f"Productos: {N_PRODUCTS}")
print(f"Pedidos: {N_ORDERS}")
print(f"Líneas de pedido: {N_ORDER_ITEMS}")

Configuración cargada
Clientes: 500
Categorías: 10
Productos: 70
Pedidos: 2000
Líneas de pedido: 4500


4. Generamos la tabla Categories

In [ ]:
category_names = [
    "Smartphones",
    "Laptops",
    "Tablets",
    "Monitors",
    "Keyboards",
    "Mice",
    "Headphones",
    "Wearables",
    "Cameras",
    "Storage"
]

category_descriptions = {
    "Smartphones": "Smartphones y teléfonos móviles de última generación.",
    "Laptops": "Ordenadores portátiles para uso personal y profesional.",
    "Tablets": "Tablets y dispositivos móviles de pantalla táctil.",
    "Monitors": "Monitores y pantallas para ordenador.",
    "Keyboards": "Teclados para ordenadores y dispositivos.",
    "Mice": "Ratones y dispositivos de navegación.",
    "Headphones": "Auriculares, cascos y dispositivos de audio.",
    "Wearables": "Smartwatches, pulseras y dispositivos tecnológicos wearables.",
    "Cameras": "Cámaras digitales y accesorios de fotografía.",
    "Storage": "Discos SSD, almacenamiento externo y tarjetas de memoria."
}


def generate_id(prefix):
    return f"{prefix}_{uuid.uuid4().hex[:8].upper()}"


df_categories = pd.DataFrame([
    {
        "category_id": generate_id("CAT"),
        "name": category_name,
        "description": category_descriptions[category_name]
    }
    for category_name in category_names
])

category_ids = df_categories["category_id"].tolist()

print(f"Categorías generadas: {len(df_categories)}")
print(f"IDs únicos: {df_categories['category_id'].nunique()}")
print()
print(df_categories)

✅ Categorías generadas: 10
✅ IDs únicos: 10

    category_id         name  \
0  CAT_AAAF0B96  Smartphones   
1  CAT_AF5FFD56      Laptops   
2  CAT_9F28ECB8      Tablets   
3  CAT_1298E8DB     Monitors   
4  CAT_4396FE3E    Keyboards   
5  CAT_0FC4447F         Mice   
6  CAT_287A92DB   Headphones   
7  CAT_F71ABD4A    Wearables   
8  CAT_FB6CDC8F      Cameras   
9  CAT_180B34BD      Storage   

                                         description  
0  Smartphones y teléfonos móviles de última gene...  
1  Ordenadores portátiles para uso personal y pro...  
2  Tablets y dispositivos móviles de pantalla tác...  
3              Monitores y pantallas para ordenador.  
4          Teclados para ordenadores y dispositivos.  
5              Ratones y dispositivos de navegación.  
6       Auriculares, cascos y dispositivos de audio.  
7  Smartwatches, pulseras y dispositivos tecnológ...  
8      Cámaras digitales y accesorios de fotografía.  
9  Discos SSD, almacenamiento externo y tarjetas ...

5. Generamos la tabla Products

In [ ]:
product_names = [
    # Smartphones
    "iPhone 15",
    "iPhone 15 Pro",
    "Samsung Galaxy S24",
    "Samsung Galaxy A55",
    "Google Pixel 9",
    "Xiaomi 14",
    "OnePlus 12",

    # Laptops
    "MacBook Air M3",
    "MacBook Pro M3",
    "Dell XPS 13",
    "Lenovo ThinkPad X1 Carbon",
    "HP Pavilion 15",
    "ASUS ZenBook 14",
    "Acer Aspire 5",

    # Tablets
    "iPad Air",
    "iPad Pro",
    "Samsung Galaxy Tab S9",
    "Samsung Galaxy Tab A9",
    "Lenovo Tab P12",
    "Xiaomi Pad 6",
    "Microsoft Surface Go",

    # Monitors
    "Dell UltraSharp 27",
    "LG UltraGear 27",
    "Samsung Odyssey G5",
    "ASUS ProArt 27",
    "BenQ GW2780",
    "AOC Gaming 24",
    "Philips 27E1",

    # Keyboards
    "Logitech MX Keys",
    "Keychron K2",
    "Razer BlackWidow V4",
    "Corsair K70",
    "Apple Magic Keyboard",
    "Logitech G915",
    "Microsoft Surface Keyboard",

    # Mice
    "Logitech MX Master 3S",
    "Logitech MX Anywhere 3S",
    "Razer DeathAdder V3",
    "Razer Basilisk V3",
    "Corsair M65 RGB",
    "Apple Magic Mouse",
    "Microsoft Bluetooth Mouse",

    # Headphones
    "Sony WH-1000XM5",
    "Apple AirPods Pro 2",
    "Bose QuietComfort",
    "Sennheiser Momentum 4",
    "JBL Live 770NC",
    "Samsung Galaxy Buds3",
    "Logitech G Pro X",

    # Wearables
    "Apple Watch Series 10",
    "Samsung Galaxy Watch 7",
    "Google Pixel Watch 3",
    "Garmin Venu 3",
    "Xiaomi Smart Band 9",
    "Fitbit Charge 6",
    "Huawei Watch GT 5",

    # Cameras
    "Sony Alpha A6400",
    "Canon EOS R10",
    "Nikon Z50",
    "Fujifilm X-S20",
    "GoPro HERO 12",
    "DJI Osmo Pocket 3",
    "Canon PowerShot G7 X",

    # Storage
    "Samsung T7 SSD",
    "SanDisk Extreme SSD",
    "WD My Passport SSD",
    "Kingston XS1000 SSD",
    "Crucial X9 SSD",
    "SanDisk Ultra 256GB",
    "Samsung EVO 512GB"
]

assert len(product_names) == N_PRODUCTS
assert len(set(product_names)) == N_PRODUCTS

category_map = dict(
    zip(
        df_categories["name"],
        df_categories["category_id"]
    )
)

product_category_names = (
    ["Smartphones"] * 7 +
    ["Laptops"] * 7 +
    ["Tablets"] * 7 +
    ["Monitors"] * 7 +
    ["Keyboards"] * 7 +
    ["Mice"] * 7 +
    ["Headphones"] * 7 +
    ["Wearables"] * 7 +
    ["Cameras"] * 7 +
    ["Storage"] * 7
)

products_data = []

for product_name, category_name in zip(
    product_names,
    product_category_names
):
    # Generamos primero el coste
    cost_price = round(
        Decimal(str(random.uniform(15, 1000))),
        2
    )

    # Margen comercial entre 20% y 80%
    margin = Decimal(
        str(random.uniform(1.20, 1.80))
    )

    sale_price = round(
        cost_price * margin,
        2
    )

    products_data.append({
        "product_id": generate_id("PROD"),
        "category_id": category_map[category_name],
        "name": product_name,
        "description": fake.text(max_nb_chars=120),
        "sale_price": sale_price,
        "cost_price": cost_price,
        "stock": random.randint(0, 500),
        "is_active": random.random() < 0.90
    })

df_products = pd.DataFrame(products_data)

print(f"Productos generados: {len(df_products)}")
print(f"IDs únicos: {df_products['product_id'].nunique()}")
print(f"Categorías utilizadas: {df_products['category_id'].nunique()}")

# Comprobaciones
invalid_prices = df_products[
    df_products["sale_price"] <= df_products["cost_price"]
]

print(
    f"Productos con precio de venta incorrecto: "
    f"{len(invalid_prices)}"
)

print()
print(
    df_products[
        [
            "product_id",
            "category_id",
            "name",
            "sale_price",
            "cost_price",
            "stock",
            "is_active"
        ]
    ].head(10)
)

✅ Productos generados: 70
✅ IDs únicos: 70
✅ Categorías utilizadas: 10
✅ Productos con precio de venta incorrecto: 0

      product_id   category_id                name sale_price cost_price  \
0  PROD_C4304C50  CAT_AAAF0B96           iPhone 15      61.10      43.59   
1  PROD_E4DDC609  CAT_AAAF0B96       iPhone 15 Pro    1241.28     728.32   
2  PROD_6305A99C  CAT_AAAF0B96  Samsung Galaxy S24     763.66     627.92   
3  PROD_0C8D6F89  CAT_AAAF0B96  Samsung Galaxy A55     835.31     668.05   
4  PROD_CC4971FF  CAT_AAAF0B96      Google Pixel 9     577.90     469.77   
5  PROD_EAF82F1F  CAT_AAAF0B96           Xiaomi 14     649.19     462.88   
6  PROD_1C7714EB  CAT_AAAF0B96          OnePlus 12    1284.10     755.73   
7  PROD_665CB463  CAT_AF5FFD56      MacBook Air M3     788.83     478.25   
8  PROD_6D450F25  CAT_AF5FFD56      MacBook Pro M3     571.97     359.71   
9  PROD_3D32E45A  CAT_AF5FFD56         Dell XPS 13      66.60      54.65   

   stock  is_active  
0    347       True  
1

Usaremos esta función para generar fechas realistas de los pedidos y de las reseñas

In [ ]:
def random_date(start_date, end_date):
    days = (end_date - start_date).days
    return start_date + timedelta(
        days=random.randint(0, days)
    )

print("Funciones auxiliares cargadas")

✅ Funciones auxiliares cargadas


6. Generamos la tabla Customers

In [ ]:
countries_cities = {
    "Spain": [
        "Madrid",
        "Barcelona",
        "Valencia",
        "Seville",
        "Bilbao"
    ],
    "France": [
        "Paris",
        "Lyon",
        "Marseille",
        "Toulouse"
    ],
    "Italy": [
        "Rome",
        "Milan",
        "Naples",
        "Turin"
    ],
    "Germany": [
        "Berlin",
        "Munich",
        "Hamburg",
        "Cologne"
    ],
    "Portugal": [
        "Lisbon",
        "Porto",
        "Braga"
    ],
    "United Kingdom": [
        "London",
        "Manchester",
        "Liverpool",
        "Birmingham"
    ]
}

reachout_channels = [
    "Organic",
    "Social Media",
    "Google Ads",
    "Email",
    "Referral",
    "Direct"
]

customers_data = []

for _ in range(N_CUSTOMERS):
    country = random.choice(list(countries_cities.keys()))
    city = random.choice(countries_cities[country])

    customers_data.append({
        "customer_id": generate_id("CUST"),
        "first_name": fake.first_name(),
        "last_name": fake.last_name(),
        "email": fake.unique.email(),
        "phone": fake.phone_number(),
        "country": country,
        "city": city,
        "reachout_channel": random.choice(reachout_channels),
        "registration_date": random_date(
            date(2022, 1, 1),
            date(2026, 8, 31)
        )
    })

df_customers = pd.DataFrame(customers_data)

print(f"Clientes generados: {len(df_customers)}")
print(f"IDs únicos: {df_customers['customer_id'].nunique()}")
print(f"Emails únicos: {df_customers['email'].nunique()}")
print(f"Países: {df_customers['country'].nunique()}")
print(f"Ciudades: {df_customers['city'].nunique()}")

print()
print(df_customers.head(10))

✅ Clientes generados: 500
✅ IDs únicos: 500
✅ Emails únicos: 500
✅ Países: 6
✅ Ciudades: 24

     customer_id first_name   last_name                         email  \
0  CUST_7B8F025D     Tomasa      Mariño        baudelio05@example.org   
1  CUST_08D565FC    Lorenza        Piña            ruth98@example.org   
2  CUST_7C7041BA   Fabricio    Salmerón         lunaroura@example.com   
3  CUST_74DA79AB     Olalla  Echeverría  clotildetrujillo@example.net   
4  CUST_BDEF5308  Godofredo       Baños     calixtoroldan@example.org   
5  CUST_4AC69446   Cornelio        Pina          halmansa@example.org   
6  CUST_4A1BD5EE     Albert      Agustí       nevadoreyna@example.net   
7  CUST_E37E1DB5     Marino      Mármol           pcortes@example.org   
8  CUST_DEB7F63E   Candelas      Martin       vidalsoraya@example.com   
9  CUST_963E4D4F    Eugenio       Osuna           jaime61@example.com   

              phone         country        city reachout_channel  \
0      +34886240891           Italy

7. Generamos la tabla Orders

In [ ]:
order_statuses = [
    "Pending",
    "Processing",
    "Shipped",
    "Delivered",
    "Cancelled"
]

orders_data = []

customer_lookup = df_customers.set_index("customer_id").to_dict("index")

for _ in range(N_ORDERS):

    customer_id = random.choice(
        df_customers["customer_id"].tolist()
    )

    customer = customer_lookup[customer_id]

    # El pedido no puede ser anterior al registro del cliente
    registration_date = customer["registration_date"]

    order_date = random_date(
        registration_date,
        date(2026, 8, 31)
    )

    order_status = random.choice(order_statuses)

    shipped_date = None
    delivered_date = None

    # Fechas posteriores al pedido según el estado
    if order_status in ["Shipped", "Delivered"]:
        shipped_date = order_date + timedelta(
            days=random.randint(1, 5)
        )

    if order_status == "Delivered":
        delivered_date = shipped_date + timedelta(
            days=random.randint(1, 7)
        )

    orders_data.append({
        "order_id": generate_id("ORD"),
        "customer_id": customer_id,
        "order_status": order_status,
        "shipping_address": fake.street_address(),
        "shipping_city": customer["city"],
        "shipping_country": customer["country"],
        "order_date": order_date,
        "shipped_date": shipped_date,
        "delivered_date": delivered_date
    })

df_orders = pd.DataFrame(orders_data)

print(f"Pedidos generados: {len(df_orders)}")
print(f"IDs únicos: {df_orders['order_id'].nunique()}")
print(f"Clientes con pedidos: {df_orders['customer_id'].nunique()}")
print()
print("Estados:")
print(df_orders["order_status"].value_counts())
print()
print(df_orders.head(10))

✅ Pedidos generados: 2000
✅ IDs únicos: 2000
✅ Clientes con pedidos: 487

Estados:
order_status
Delivered     415
Cancelled     406
Pending       403
Processing    389
Shipped       387
Name: count, dtype: int64

       order_id    customer_id order_status  \
0  ORD_6F94F73B  CUST_A8621156    Cancelled   
1  ORD_8A6E08C1  CUST_A8B865B4      Pending   
2  ORD_9462E694  CUST_AF7BEFD2      Shipped   
3  ORD_FA5CBCA2  CUST_A58D370E      Pending   
4  ORD_206D8236  CUST_811768DE      Pending   
5  ORD_8025AC67  CUST_31D4ACBB    Cancelled   
6  ORD_6B8CE2FB  CUST_20270000    Delivered   
7  ORD_5FE54230  CUST_1F966EBF   Processing   
8  ORD_BDD174A6  CUST_B0D97172    Delivered   
9  ORD_275F9C8E  CUST_CC3CFFF5      Pending   

                           shipping_address shipping_city shipping_country  \
0       Acceso de Nicolás Acedo 8 Puerta 3        Cologne          Germany   
1                   Alameda Irma Vazquez 78        London   United Kingdom   
2              Calle de Wálter Manr

8. Generamos la tabla Order_items

In [ ]:
order_items_data = []

product_lookup = (
    df_products
    .set_index("product_id")
    .to_dict("index")
)

order_ids = df_orders["order_id"].tolist()
product_ids = df_products["product_id"].tolist()

for _ in range(N_ORDER_ITEMS):

    order_id = random.choice(order_ids)
    product_id = random.choice(product_ids)

    product = product_lookup[product_id]

    # Precio histórico de compra:
    # puede diferir ligeramente del precio actual
    current_price = Decimal(
        str(product["sale_price"])
    )

    price_variation = Decimal(
        str(random.uniform(0.90, 1.05))
    )

    unit_price = round(
        current_price * price_variation,
        2
    )

    quantity = random.randint(1, 3)

    # Descuento entre 0% y 20%
    discount = round(
        Decimal(str(random.uniform(0, 0.20))),
        2
    )

    order_items_data.append({
        "order_item_id": generate_id("ITEM"),
        "order_id": order_id,
        "product_id": product_id,
        "quantity": quantity,
        "unit_price": unit_price,
        "discount": discount
    })

df_order_items = pd.DataFrame(order_items_data)

print(f"Líneas de pedido generadas: {len(df_order_items)}")
print(
    f"IDs únicos: "
    f"{df_order_items['order_item_id'].nunique()}"
)
print(
    f"Pedidos utilizados: "
    f"{df_order_items['order_id'].nunique()}"
)
print(
    f"Productos utilizados: "
    f"{df_order_items['product_id'].nunique()}"
)
print()
print(df_order_items.head(10))

✅ Líneas de pedido generadas: 4500
✅ IDs únicos: 4500
✅ Pedidos utilizados: 1783
✅ Productos utilizados: 70

   order_item_id      order_id     product_id  quantity unit_price discount
0  ITEM_1F13E3AD  ORD_B2342694  PROD_D390B4BA         3    1052.94     0.12
1  ITEM_E6C0EE13  ORD_9E973653  PROD_EAF82F1F         1     632.13     0.07
2  ITEM_EC9248A0  ORD_9AA9D838  PROD_6050761B         3     774.91     0.06
3  ITEM_BD90C8F8  ORD_B3DDCFD4  PROD_EAF7ED80         1     207.10     0.11
4  ITEM_6B42811B  ORD_C8A4824F  PROD_5DE49720         1     222.88     0.03
5  ITEM_FEF7A849  ORD_19DA1CB4  PROD_8E554760         1    1013.71     0.12
6  ITEM_8A73256B  ORD_1914C1DB  PROD_E89ACC7F         1     930.43     0.06
7  ITEM_A75566B1  ORD_544EBED0  PROD_ED7A7DA0         2     138.30     0.05
8  ITEM_0B2F088A  ORD_018D1C18  PROD_7F651855         1    1176.56     0.15
9  ITEM_8850D7FE  ORD_B03F864F  PROD_A0E546EC         3      43.07     0.08


9. Generamos la tabla Payments

In [ ]:
payments_data = []

order_items_grouped = (
    df_order_items
    .assign(
        line_total=lambda df: (
            df["quantity"]
            * df["unit_price"]
            * (1 - df["discount"])
        )
    )
    .groupby("order_id")["line_total"]
    .sum()
)

for _, order in df_orders.iterrows():

    order_id = order["order_id"]

    # Si el pedido no tiene líneas, importe 0
    amount = order_items_grouped.get(
        order_id,
        Decimal("0.00")
    )

    amount = round(
        Decimal(str(amount)),
        2
    )

    if order["order_status"] == "Cancelled":
        payment_status = "Refunded"
    elif order["order_status"] == "Delivered":
        payment_status = "Paid"
    elif order["order_status"] in ["Shipped", "Processing"]:
        payment_status = random.choice(
            ["Paid", "Paid", "Pending"]
        )
    else:
        payment_status = random.choice(
            ["Pending", "Failed"]
        )

    payment_methods = [
        "Credit Card",
        "Debit Card",
        "PayPal",
        "Bank Transfer",
        "Apple Pay"
    ]

    payments_data.append({
        "payment_id": generate_id("PAY"),
        "order_id": order_id,
        "payment_method": random.choice(payment_methods),
        "payment_status": payment_status,
        "amount": amount,
        "payment_date": order["order_date"]
    })

df_payments = pd.DataFrame(payments_data)

print(f"Pagos generados: {len(df_payments)}")
print(
    f"IDs únicos: "
    f"{df_payments['payment_id'].nunique()}"
)
print(
    f"Pedidos con pago: "
    f"{df_payments['order_id'].nunique()}"
)
print()
print("Estados de pago:")
print(df_payments["payment_status"].value_counts())
print()
print(df_payments.head(10))

✅ Pagos generados: 2000
✅ IDs únicos: 2000
✅ Pedidos con pago: 2000

Estados de pago:
payment_status
Paid        942
Pending     441
Refunded    406
Failed      211
Name: count, dtype: int64

     payment_id      order_id payment_method payment_status   amount  \
0  PAY_416B5EBE  ORD_6F94F73B  Bank Transfer       Refunded   768.42   
1  PAY_2E4967D4  ORD_8A6E08C1         PayPal         Failed   614.02   
2  PAY_6320692E  ORD_9462E694         PayPal        Pending  3705.98   
3  PAY_EA82E467  ORD_FA5CBCA2      Apple Pay        Pending  3848.31   
4  PAY_919AB234  ORD_206D8236         PayPal         Failed  2304.18   
5  PAY_4A4FF848  ORD_8025AC67      Apple Pay       Refunded  7470.22   
6  PAY_EBEEFB1F  ORD_6B8CE2FB  Bank Transfer           Paid  6444.62   
7  PAY_336F7707  ORD_5FE54230    Credit Card           Paid  3881.86   
8  PAY_B628283C  ORD_BDD174A6    Credit Card           Paid  2321.66   
9  PAY_0F3A62DE  ORD_275F9C8E      Apple Pay         Failed  1445.81   

  payment_date 

10. Generamos la tabla Reviews

In [ ]:
reviews_data = []

CUTOFF_DATE = date(2026, 8, 31)

# Relación order_id -> customer_id
order_customer_map = dict(
    zip(
        df_orders["order_id"],
        df_orders["customer_id"]
    )
)

# Relación order_id -> delivered_date
order_delivered_map = dict(
    zip(
        df_orders["order_id"],
        df_orders["delivered_date"]
    )
)

# Solo pedidos entregados hasta la fecha de corte
delivered_orders = df_orders[
    (df_orders["order_status"] == "Delivered") &
    (df_orders["delivered_date"] <= CUTOFF_DATE)
]

delivered_order_ids = set(
    delivered_orders["order_id"]
)

# Líneas pertenecientes a esos pedidos
delivered_items = df_order_items[
    df_order_items["order_id"].isin(
        delivered_order_ids
    )
].copy()

comments = {
    1: [
        "El producto no cumplió mis expectativas.",
        "La experiencia de compra no fue buena."
    ],
    2: [
        "El producto está bien, pero esperaba más.",
        "Tiene algunos aspectos mejorables."
    ],
    3: [
        "Producto correcto.",
        "Cumple con lo esperado."
    ],
    4: [
        "Muy buen producto, estoy satisfecho.",
        "Buena calidad y funcionamiento."
    ],
    5: [
        "Excelente producto, muy recomendable.",
        "Estoy encantado con la compra.",
        "Gran calidad y muy buena experiencia."
    ]
}

# Aproximadamente 35% de las líneas entregadas
for _, item in delivered_items.iterrows():

    if random.random() > 0.35:
        continue

    order_id = item["order_id"]
    customer_id = order_customer_map[order_id]
    delivered_date = order_delivered_map[order_id]

    rating = random.choices(
        [1, 2, 3, 4, 5],
        weights=[3, 5, 12, 30, 50],
        k=1
    )[0]

    # La review se publica desde la entrega
    # hasta la fecha de corte
    review_date = random_date(
        delivered_date,
        CUTOFF_DATE
    )

    reviews_data.append({
        "review_id": generate_id("REV"),
        "order_item_id": item["order_item_id"],
        "customer_id": customer_id,
        "rating": rating,
        "comment": random.choice(
            comments[rating]
        ),
        "review_date": review_date
    })

df_reviews = pd.DataFrame(reviews_data)

print(f"Reviews generadas: {len(df_reviews)}")
print(
    f"IDs únicos: "
    f"{df_reviews['review_id'].nunique()}"
)
print(
    f"Líneas de pedido revisadas: "
    f"{df_reviews['order_item_id'].nunique()}"
)
print(
    f"Pedidos entregados hasta la fecha de corte: "
    f"{len(delivered_orders)}"
)
print()
print("Distribución de ratings:")
print(
    df_reviews["rating"]
    .value_counts()
    .sort_index()
)
print()
print(df_reviews.head(10))

✅ Reviews generadas: 317
✅ IDs únicos: 317
✅ Líneas de pedido revisadas: 317
✅ Pedidos entregados hasta la fecha de corte: 401

Distribución de ratings:
rating
1      9
2      5
3     42
4     93
5    168
Name: count, dtype: int64

      review_id  order_item_id    customer_id  rating  \
0  REV_8E4679B0  ITEM_EC9248A0  CUST_1A0D48B0       5   
1  REV_33AB7E67  ITEM_BEA1D111  CUST_2506C036       5   
2  REV_0FBB00B7  ITEM_A4892ACE  CUST_6A126426       5   
3  REV_0397454E  ITEM_631C8895  CUST_9566B01A       3   
4  REV_B579F68D  ITEM_BB27F4FC  CUST_1918D95F       5   
5  REV_BA902972  ITEM_21DF14B4  CUST_3C23795C       5   
6  REV_01DE5793  ITEM_3CB05167  CUST_25276106       5   
7  REV_7EFC3695  ITEM_F9FF8175  CUST_55E161A1       3   
8  REV_B5DC1EC7  ITEM_85E465C7  CUST_7C819123       5   
9  REV_4F45D209  ITEM_ACD48640  CUST_CCDFB60E       5   

                                 comment review_date  
0         Estoy encantado con la compra.  2026-08-18  
1  Gran calidad y muy buena ex

11. Mostramos un resumen de todos los DataFrames que hemos creado

In [ ]:
dataframes = {
    "customers": df_customers,
    "categories": df_categories,
    "products": df_products,
    "orders": df_orders,
    "order_items": df_order_items,
    "payments": df_payments,
    "reviews": df_reviews
}

print("RESUMEN DE DATOS GENERADOS")

for table_name, df in dataframes.items():
    print(
        f"{table_name:15} -> "
        f"{len(df):5} filas | "
        f"{len(df.columns):2} columnas"
    )



RESUMEN DE DATOS GENERADOS
customers       ->   500 filas |  9 columnas
categories      ->    10 filas |  3 columnas
products        ->    70 filas |  8 columnas
orders          ->  2000 filas |  9 columnas
order_items     ->  4500 filas |  6 columnas
payments        ->  2000 filas |  6 columnas
reviews         ->   317 filas |  6 columnas


12. Validamos las claves foráneas, comprobamos que los IDs que aparecen como referencias en una tabla realmente existen en la tabla a la que apuntan.

In [ ]:
print("VALIDACIÓN DE CLAVES FORÁNEAS")

# IDs válidos de cada tabla
category_ids = set(df_categories["category_id"])
customer_ids = set(df_customers["customer_id"])
product_ids = set(df_products["product_id"])
order_ids = set(df_orders["order_id"])
order_item_ids = set(df_order_items["order_item_id"])


# 1. PRODUCTS → CATEGORIES
invalid_product_categories = (
    ~df_products["category_id"].isin(category_ids)
).sum()

print(
    f"products → categories: "
    f"{invalid_product_categories} referencias inválidas"
)


# 2. ORDERS → CUSTOMERS
invalid_order_customers = (
    ~df_orders["customer_id"].isin(customer_ids)
).sum()

print(
    f"orders → customers: "
    f"{invalid_order_customers} referencias inválidas"
)


# 3. ORDER_ITEMS → ORDERS
invalid_item_orders = (
    ~df_order_items["order_id"].isin(order_ids)
).sum()

print(
    f"order_items → orders: "
    f"{invalid_item_orders} referencias inválidas"
)


# 4. ORDER_ITEMS → PRODUCTS
invalid_item_products = (
    ~df_order_items["product_id"].isin(product_ids)
).sum()

print(
    f"order_items → products: "
    f"{invalid_item_products} referencias inválidas"
)


# 5. PAYMENTS → ORDERS
invalid_payment_orders = (
    ~df_payments["order_id"].isin(order_ids)
).sum()

print(
    f"payments → orders: "
    f"{invalid_payment_orders} referencias inválidas"
)


# 6. REVIEWS → ORDER_ITEMS
invalid_review_items = (
    ~df_reviews["order_item_id"].isin(order_item_ids)
).sum()

print(
    f"reviews → order_items: "
    f"{invalid_review_items} referencias inválidas"
)


# 7. REVIEWS → CUSTOMERS
invalid_review_customers = (
    ~df_reviews["customer_id"].isin(customer_ids)
).sum()

print(
    f"reviews → customers: "
    f"{invalid_review_customers} referencias inválidas"
)


print("=" * 60)

VALIDACIÓN DE CLAVES FORÁNEAS
products → categories: 0 referencias inválidas
orders → customers: 0 referencias inválidas
order_items → orders: 0 referencias inválidas
order_items → products: 0 referencias inválidas
payments → orders: 0 referencias inválidas
reviews → order_items: 0 referencias inválidas
reviews → customers: 0 referencias inválidas


13. Validamos las claves primarias para comprobar que cada registro tiene un ID único

In [ ]:
print("VALIDACIÓN DE CLAVES PRIMARIAS")

primary_keys = {
    "customers": ("customer_id", df_customers),
    "categories": ("category_id", df_categories),
    "products": ("product_id", df_products),
    "orders": ("order_id", df_orders),
    "order_items": ("order_item_id", df_order_items),
    "payments": ("payment_id", df_payments),
    "reviews": ("review_id", df_reviews)
}

all_valid = True

for table_name, (pk, df) in primary_keys.items():

    total_rows = len(df)
    unique_values = df[pk].nunique()
    duplicates = total_rows - unique_values

    if duplicates == 0:
        print(
            f"{table_name:15} "
            f"{pk:15} → sin duplicados"
        )
    else:
        print(
            f"{table_name:15} "
            f"{pk:15} → {duplicates} duplicados"
        )
        all_valid = False


if all_valid:
    print("Todas las claves primarias son únicas")
else:
    print("Hay claves primarias duplicadas")

VALIDACIÓN DE CLAVES PRIMARIAS
✅ customers       customer_id     → sin duplicados
✅ categories      category_id     → sin duplicados
✅ products        product_id      → sin duplicados
✅ orders          order_id        → sin duplicados
✅ order_items     order_item_id   → sin duplicados
✅ payments        payment_id      → sin duplicados
✅ reviews         review_id       → sin duplicados
🎉 Todas las claves primarias son únicas


14. Compramos que se cumplen las reglas de negocio que están establecidas

In [ ]:
print("VALIDACIÓN DE REGLAS DE NEGOCIO")

errors = 0


# 1. Precio de venta > coste
invalid_product_prices = (
    df_products["sale_price"] <= df_products["cost_price"]
).sum()

print(
    f"Productos con sale_price <= cost_price: "
    f"{invalid_product_prices}"
)

errors += invalid_product_prices


# 2. Stock >= 0
invalid_stock = (
    df_products["stock"] < 0
).sum()

print(
    f"Productos con stock negativo: "
    f"{invalid_stock}"
)

errors += invalid_stock


# 3. Cantidad de productos > 0
invalid_quantities = (
    df_order_items["quantity"] <= 0
).sum()

print(
    f"Líneas con cantidad <= 0: "
    f"{invalid_quantities}"
)

errors += invalid_quantities


# 4. Descuento entre 0 y 20%
invalid_discounts = (
    (df_order_items["discount"] < 0) |
    (df_order_items["discount"] > 0.20)
).sum()

print(
    f"Líneas con descuento fuera de 0%-20%: "
    f"{invalid_discounts}"
)

errors += invalid_discounts


# 5. Ratings entre 1 y 5
invalid_ratings = (
    (df_reviews["rating"] < 1) |
    (df_reviews["rating"] > 5)
).sum()

print(
    f"Reviews con rating fuera de 1-5: "
    f"{invalid_ratings}"
)

errors += invalid_ratings


# 6. Pedidos: shipped_date >= order_date
invalid_shipping_dates = (
    df_orders["shipped_date"].notna() &
    (
        df_orders["shipped_date"] <
        df_orders["order_date"]
    )
).sum()

print(
    f"Pedidos con shipped_date anterior a order_date: "
    f"{invalid_shipping_dates}"
)

errors += invalid_shipping_dates


# 7. Pedidos: delivered_date >= shipped_date
invalid_delivery_dates = (
    df_orders["delivered_date"].notna() &
    (
        df_orders["shipped_date"].isna() |
        (
            df_orders["delivered_date"] <
            df_orders["shipped_date"]
        )
    )
).sum()

print(
    f"Pedidos con delivered_date incorrecta: "
    f"{invalid_delivery_dates}"
)

errors += invalid_delivery_dates


# 8. Reviews posteriores a la entrega
review_order_dates = df_reviews[
    ["review_id", "order_item_id", "review_date"]
].merge(
    df_order_items[
        ["order_item_id", "order_id"]
    ],
    on="order_item_id",
    how="left"
).merge(
    df_orders[
        ["order_id", "delivered_date"]
    ],
    on="order_id",
    how="left"
)

invalid_review_dates = (
    review_order_dates["review_date"] <
    review_order_dates["delivered_date"]
).sum()

print(
    f"Reviews anteriores a la entrega: "
    f"{invalid_review_dates}"
)

errors += invalid_review_dates



if errors == 0:
    print("TODAS LAS REGLAS DE NEGOCIO SON VÁLIDAS")
else:
    print(
        f"Se han encontrado {errors} "
        f"errores de validación"
    )

VALIDACIÓN DE REGLAS DE NEGOCIO
Productos con sale_price <= cost_price: 0
Productos con stock negativo: 0
Líneas con cantidad <= 0: 0
Líneas con descuento fuera de 0%-20%: 0
Reviews con rating fuera de 1-5: 0
Pedidos con shipped_date anterior a order_date: 0
Pedidos con delivered_date incorrecta: 0
Reviews anteriores a la entrega: 0
🎉 TODAS LAS REGLAS DE NEGOCIO SON VÁLIDAS


15. Cargamos todos los DataFrames en un BigQuery

In [ ]:
tables_to_load = {
    "customers": df_customers,
    "categories": df_categories,
    "products": df_products,
    "orders": df_orders,
    "order_items": df_order_items,
    "payments": df_payments,
    "reviews": df_reviews
}

print("CARGANDO DATOS EN BIGQUERY")

for table_name, df in tables_to_load.items():

    table_id = f"{PROJECT_ID}.{DATASET_ID}.{table_name}"

    job_config = bigquery.LoadJobConfig(
        write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE
    )

    job = client.load_table_from_dataframe(
        df,
        table_id,
        job_config=job_config
    )

    job.result()

    table = client.get_table(table_id)

    print(
        f"{table_name:15} "
        f"→ {table.num_rows:,} filas cargadas"
    )

print("CARGA COMPLETADA")


CARGANDO DATOS EN BIGQUERY


c:\Users\casco\Desktop\bootcamp\Entregas\SQLMurderMystery\.venv\Lib\site-packages\google\cloud\bigquery\_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


✅ customers       → 500 filas cargadas


c:\Users\casco\Desktop\bootcamp\Entregas\SQLMurderMystery\.venv\Lib\site-packages\google\cloud\bigquery\_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


✅ categories      → 10 filas cargadas


...

✅ order_items     → 4,500 filas cargadas


c:\Users\casco\Desktop\bootcamp\Entregas\SQLMurderMystery\.venv\Lib\site-packages\google\cloud\bigquery\_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


✅ payments        → 2,000 filas cargadas


c:\Users\casco\Desktop\bootcamp\Entregas\SQLMurderMystery\.venv\Lib\site-packages\google\cloud\bigquery\_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


✅ reviews         → 317 filas cargadas
🎉 CARGA COMPLETADA


16. Realizamos la verificación final para comprobar que los datos que tenemos en Python son iguales a los que hemos cargado en BigQuery

In [ ]:
print("VERIFICACIÓN DIRECTA EN BIGQUERY")

for table_name, df in tables_to_load.items():

    query = f"""
        SELECT COUNT(*) AS total
        FROM `{PROJECT_ID}.{DATASET_ID}.{table_name}`
    """

    result = client.query(query).result()
    total_bq = list(result)[0].total
    total_python = len(df)

    if total_bq == total_python:
        print(
            f"{table_name:15} "
            f"Python: {total_python:,} | "
            f"BigQuery: {total_bq:,}"
        )
    else:
        print(
            f"{table_name:15} "
            f"Python: {total_python:,} | "
            f"BigQuery: {total_bq:,}"
        )

print("VERIFICACIÓN FINALIZADA")


VERIFICACIÓN DIRECTA EN BIGQUERY
✅ customers       Python: 500 | BigQuery: 500
✅ categories      Python: 10 | BigQuery: 10
✅ products        Python: 70 | BigQuery: 70
✅ orders          Python: 2,000 | BigQuery: 2,000
✅ order_items     Python: 4,500 | BigQuery: 4,500
✅ payments        Python: 2,000 | BigQuery: 2,000
✅ reviews         Python: 317 | BigQuery: 317
🎉 VERIFICACIÓN FINALIZADA
